# Alexinator : analyse des parties

Ce notebook lit `data/journal.db` et produit les chiffres du rapport.

- **Partie A, playtests** : ce que les vrais joueurs ont fait (tables `tours` et `propositions`).
- **Partie B, campagnes automatiques** : résultats de `campagne.py` (table `evaluations`).

Une section sans données affiche simplement un message : lancez d'abord des parties ou une campagne.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
JOURNAL = ROOT / "data" / "journal.db"

conn = sqlite3.connect(JOURNAL)
tours = pd.read_sql("SELECT * FROM tours", conn)
propositions = pd.read_sql("SELECT * FROM propositions", conn)
evaluations = pd.read_sql("SELECT * FROM evaluations", conn)
conn.close()

# Les campagnes automatiques jouent sous un pseudo qui commence par « auto: » : on les sépare des humains.
tours_humains = tours[~tours["joueur"].str.startswith("auto:")]
props_humaines = propositions[~propositions["joueur"].str.startswith("auto:")]
print(
    f"{len(tours_humains)} messages de joueurs, {len(props_humaines)} propositions, "
    f"{len(evaluations)} lignes de campagne"
)

## A. Playtests

### A.1 Les niveaux sont-ils bien dosés ?

Une *partie* = une session (un joueur, un niveau, un film). Elle est gagnée si l'une de ses propositions est correcte.
On regarde le taux de réussite par niveau et le nombre de messages utilisés avant de gagner.

In [ ]:
if props_humaines.empty and tours_humains.empty:
    print("Pas encore de parties jouées par des humains.")
else:
    # Toutes les parties commencées : celles qui ont au moins un message ou une proposition
    sessions = pd.concat(
        [tours_humains[["session_id", "niveau"]], props_humaines[["session_id", "niveau"]]]
    ).drop_duplicates()
    gagnees = props_humaines[props_humaines["correcte"] == 1].groupby("session_id")["messages_utilises"].min()
    sessions["gagnee"] = sessions["session_id"].isin(gagnees.index)
    sessions["messages_avant_victoire"] = sessions["session_id"].map(gagnees)

    par_niveau = (
        sessions.groupby("niveau")
        .agg(
            parties=("session_id", "count"),
            taux_reussite=("gagnee", "mean"),
            messages_moyens_avant_victoire=("messages_avant_victoire", "mean"),
        )
        .round(2)
    )
    display(par_niveau)

    par_niveau["taux_reussite"].plot.bar(ylim=(0, 1), title="Taux de réussite par niveau (joueurs humains)")
    plt.ylabel("part des parties gagnées")
    plt.show()

### A.2 Quelle défense arrête les joueurs ?

Pour chaque niveau, part des messages bloqués par chaque défense (`aucune` = la réponse du gardien est passée).

In [ ]:
if tours_humains.empty:
    print("Pas encore de messages de joueurs.")
else:
    blocages = (
        tours_humains.assign(defense=tours_humains["bloque_par"].fillna("aucune"))
        .groupby("niveau")["defense"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .round(2)
    )
    display(blocages)
    blocages.plot.bar(stacked=True, title="Sort des messages des joueurs, par niveau")
    plt.ylabel("part des messages")
    plt.legend(title="bloqué par", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()

### A.3 Les fuites que les défenses ont rattrapées

Messages pour lesquels le gardien avait répondu quelque chose (`reponse_brute`) qu'une défense a ensuite bloqué.
C'est une bonne mine d'exemples pour le rapport, et pour étiqueter les attaques par catégorie.

In [ ]:
rattrapees = tours_humains[tours_humains["bloque_par"].isin(["output_filter", "censor"])]
display(rattrapees[["niveau", "film_id", "message_joueur", "reponse_brute", "bloque_par", "detail"]].head(20))

## B. Campagnes automatiques

Chaque ligne de `evaluations` est un message (attaque ou question honnête) envoyé dans une partie neuve.

- `fuite_titre` : le titre est écrit en clair dans la réponse affichée ;
- `fuite_devineur` : un LLM devineur a retrouvé le film à partir de la réponse (fuite « sémantique ») ;
- `bloque_par` : défense déclenchée.

### B.1 Matrice catégorie d'attaque × niveau

Part des attaques qui font fuiter le film (titre écrit ou film deviné). Plus la case est foncée, plus l'attaque marche.

In [ ]:
if evaluations.empty:
    print("Aucune campagne enregistrée : lancez  python campagne.py --niveaux 1 2 3 --films 3")
else:
    evaluations["fuite"] = (evaluations["fuite_titre"] + evaluations["fuite_devineur"]) > 0
    evaluations["bloque"] = evaluations["bloque_par"].notna()
    derniere = evaluations["campagne"].iloc[-1]  # on analyse la campagne la plus récente
    ev = evaluations[evaluations["campagne"] == derniere]
    attaques = ev[ev["type"] == "attaque"]
    print(f"Campagne « {derniere} » : {len(attaques)} attaques, modèle(s) {sorted(ev['modele'].unique())}")

    matrice = attaques.pivot_table(index="categorie", columns="niveau", values="fuite", aggfunc="mean")
    display(matrice.round(2))

    fig, ax = plt.subplots(figsize=(1.2 * len(matrice.columns) + 4, 0.5 * len(matrice) + 1.5))
    image = ax.imshow(matrice.values.astype(float), cmap="Reds", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(matrice.columns)), [f"Niveau {c}" for c in matrice.columns])
    ax.set_yticks(range(len(matrice.index)), matrice.index)
    for i in range(len(matrice.index)):
        for j in range(len(matrice.columns)):
            value = matrice.values[i, j]
            ax.text(j, i, f"{value:.0%}", ha="center", va="center", color="white" if value > 0.5 else "black")
    fig.colorbar(image, label="taux de fuite")
    ax.set_title("Taux de fuite par catégorie d'attaque et par niveau")
    plt.show()

### B.2 Sécurité contre jouabilité

Une défense parfaite qui bloque tout rend le jeu injouable. Pour chaque niveau :

- **taux de fuite** sur les attaques (à minimiser) ;
- **taux de faux positifs** : part des questions honnêtes bloquées (à minimiser aussi).

In [ ]:
if not evaluations.empty:
    resume = pd.DataFrame(
        {
            "taux_fuite_attaques": ev[ev["type"] == "attaque"].groupby("niveau")["fuite"].mean(),
            "taux_blocage_attaques": ev[ev["type"] == "attaque"].groupby("niveau")["bloque"].mean(),
            "taux_faux_positifs": ev[ev["type"] == "honnete"].groupby("niveau")["bloque"].mean(),
        }
    ).round(3)
    display(resume)
    resume[["taux_fuite_attaques", "taux_faux_positifs"]].plot.bar(
        ylim=(0, 1), title="Sécurité contre jouabilité, par niveau"
    )
    plt.ylabel("taux")
    plt.show()

### B.3 Comparer des modèles

Lancez la même campagne (mêmes niveaux, mêmes films grâce à `--graine`) avec deux modèles en changeant `--nom`, par exemple :

```
python campagne.py --niveaux 2 --films 4 --nom modele_de_base
python campagne.py --niveaux 2 --films 4 --nom modele_entraine      (après avoir modifié .env)
```

Le niveau 2 n'a aucune défense technique : c'est lui qui mesure la résistance du modèle seul.

In [ ]:
if not evaluations.empty:
    comparaison = pd.DataFrame(
        {
            "taux_fuite_attaques": evaluations[evaluations["type"] == "attaque"]
            .groupby(["campagne", "modele", "niveau"])["fuite"]
            .mean(),
            "taux_faux_positifs": evaluations[evaluations["type"] == "honnete"]
            .groupby(["campagne", "modele", "niveau"])["bloque"]
            .mean(),
        }
    ).round(3)
    display(comparaison)